In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8a5255b0fce49fc730baae6ea6ecc601d45bb16b"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8a5255b0fce49fc730baae6ea6ecc601d45bb16b/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Reproducibility and Inspection

Suppose we change one line of a training script and the validation accuracy
of a Fashion-MNIST classifier rises by 0.3 points. Before we credit the
change, we need to know how much two runs of the unchanged script differ.
For the classifiers of that section and
that section, they differ by a few tenths of a point, because
each run draws different initial weights and visits the examples in a
different order. A run that fails is a second problem of the same kind. If
the loss turns into NaN after hours of training, we want to repeat the run
exactly up to that point and then look inside the model to see which layer
produced the first non-finite number.

This section first makes runs repeatable. Every random draw comes from a
generator whose sequence of numbers is fixed by a seed, so seeding the
generators fixes the draws; repeating a run bit for bit also requires that
the arithmetic adds numbers in a fixed order. The section then looks inside
running models with *hooks*, functions that observe each layer of an
unmodified model as it computes. The two halves depend on each other: a
measurement taken inside a model can be compared across two versions of the
code only if the runs differ in nothing but that code.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

## Seeds and Random Number Generators

Randomness enters a training run in more places than one might list at
first: initialization draws every weight (that section), dropout
draws a new mask at every step, the data loader shuffles the examples
differently in every epoch, data augmentation draws crops and flips, and
parallel loader workers do some of this work in separate processes. Each
draw comes from a *pseudorandom number generator*, a deterministic algorithm
whose sequence of outputs is fixed by its starting state, the *seed*.
Seeding every generator that a run consults therefore fixes every draw.

### Global Seeds

Where the random functions get their state from is the first thing to know
about a framework: a global generator that they consult unless told
otherwise, or keys passed explicitly.

JAX has no global generator. Every random function takes a *key*, a value
that determines its output: `jax.random.normal(key, shape)` returns the same
numbers every time it is called with the same key, and new randomness comes
only from deriving new keys with `jax.random.split`. The function below
turns its seed into one key and splits it three ways, one key each for
initialization, inputs, and targets, so the seed fixes all its draws:

In [2]:
def train_once(seed):
    key_init, key_X, key_y = jax.random.split(jax.random.key(seed), 3)
    rngs = nnx.Rngs(key_init)
    net = nnx.Sequential(nnx.Linear(20, 32, rngs=rngs),
                         nnx.relu,
                         nnx.Linear(32, 1, rngs=rngs))
    X = jax.random.normal(key_X, (128, 20))
    y = jax.random.normal(key_y, (128, 1))
    optimizer = nnx.Optimizer(net, optax.sgd(0.1), wrt=nnx.Param)
    loss_fn = lambda model: ((model(X) - y) ** 2).mean()
    for _ in range(5):
        loss, grads = nnx.value_and_grad(loss_fn)(net)
        optimizer.update(net, grads)
    return loss, net.layers[0].kernel[...]

loss_a, w_a = train_once(seed=0)
loss_b, w_b = train_once(seed=0)
loss_c, _ = train_once(seed=1)
print('same seed, identical loss and weights:',
      bool(loss_a == loss_b), bool(jnp.array_equal(w_a, w_b)))
print('different seed:', loss_a, 'vs', loss_c)

same seed, identical loss and weights: True True
different seed: 1.0159674 vs 0.79100513


The two runs with seed 0 agree *bitwise*, down to the last bit of the loss
and of every weight: same initial weights, same data, same gradients, same
operations in the same order. The run with seed 1 ends with a different
loss.


### Generator Objects

A single global generator couples all the code that draws from it. Inserting
one extra random call, a new layer's initialization or a stray draw while
debugging, shifts every number drawn after it, so the data split, the
initialization, and the dropout masks all change at once. A *generator
object* is a private stream with its own seed, which unrelated draws cannot
shift.

In JAX the keys are the generator objects: each key is a private stream that
we hold as a value, and `jax.random.split` produces as many independent
streams as we need. No draw changes any shared state, so unrelated draws
cannot shift ours. Here a draw with another key between two uses of the
same key changes nothing:

In [3]:
key_split, key_other = jax.random.split(jax.random.key(42))
split = jax.random.permutation(key_split, 10)
_ = jax.random.normal(key_other, (1000,))  # unrelated draws, their own key
split_again = jax.random.permutation(key_split, 10)
print(bool(jnp.array_equal(split, split_again)), split)

True [1 3 9 5 0 2 6 7 4 8]


Every function in `jax.random` takes the key as its first argument; no
variant consults hidden state.

### Data Loader Workers

Data loaders often prepare minibatches in parallel *worker* processes. A
worker that draws random numbers, for data augmentation for instance, needs
a stream of its own. Two failures are common. If every worker starts as a copy of the
main process, as with the *fork* start method, Linux's default before
Python 3.14, the workers inherit the same generator
state and apply the same "random" augmentations. If each worker seeds itself
from the operating system's entropy, no two runs agree.

With explicit keys, the workers' streams appear in the program: a worker's
randomness is whatever key we hand it. We split one key into per-worker keys
with `jax.random.split(key, num_workers)` and derive new ones in each epoch
with `jax.random.fold_in(key, epoch)`, which mixes the epoch number into the
key. Two workers can share a stream only if the code passes them the same
key. JAX programs often borrow their input pipeline from PyTorch or
`tf.data`; such loaders bring their own rules for seeding workers, given in
the tabs of those frameworks.

### Randomness as a Value

The worker failures above arise because the state of a generator is copied
or shared without appearing in the program's data flow. JAX makes randomness
a value instead: every random function takes a key, the same key gives the
same numbers, and independent streams come from splitting a key. A key can
still be reused by mistake, but the reuse is visible in the code. The cost
is that every function that draws random numbers must accept a key and pass
split keys on. A global seed is the convenient form of the same contract: one
implicit key that all code shares. Generator objects recover the explicit
form where it matters.

The two cells above use this design: `train_once` turned one seed into one
key and split it, and reusing a key reproduced the permutation.

## Deterministic Arithmetic

Seeding fixes which numbers the program draws. It does not fix how the
arithmetic combines them. Floating-point addition is not associative
(that section): every sum is rounded to the nearest representable
number, so adding the same numbers in a different order can give a different
result.

The smallest example takes three float32 numbers. Near $10^8$, adjacent
float32 numbers are 8 apart, so $-10^8 + 1$ rounds back to $-10^8$:

In [4]:
a, b, c = [jnp.array(v, dtype=jnp.float32) for v in (1e8, -1e8, 1.0)]
print(float((a + b) + c), float(a + (b + c)))  # same numbers, regrouped

1.0 0.0


Regrouping changes the result from 1 to 0. In a long sum the effect is
smaller but of the same kind: adding a million random numbers forward and
backward usually gives results that differ in their last bits.

On a CPU, the order in which one operation adds its numbers is usually fixed
for a given number of threads, so a seeded run repeats. On a GPU, many
threads add partial sums at the same time, and some operations combine them
with *atomic additions*, which commit each partial sum in whatever order the
threads happen to finish. Examples are the gradients of an embedding lookup
or of a scatter operation, some convolution gradients, and some reductions.
Two seeded runs on the same machine can then differ in the last bits of
their weights, and over many training steps such differences grow until the
loss curves differ visibly. The remedy is a switch that replaces such
operations with deterministic ones, which add in a fixed order and are often
slower.

JAX's random numbers are reproducible across devices: a draw is a function
of the key and the shape only, so the same key gives the same numbers on
CPU, GPU, and TPU (JAX does not promise the same bits across its own
releases). For arithmetic, the operations of `train_once` added in a fixed
order on the GPU too. Setting the environment variable
`XLA_FLAGS=--xla_gpu_deterministic_ops=true` before JAX starts replaces
GPU operations that add in a varying order with deterministic ones; it
does not raise errors.

Bitwise agreement holds only for one machine, one library version, and one
configuration of these switches; no framework promises it across releases,
across hardware, or between CPU and GPU. Bitwise reproducibility is
therefore a tool for debugging: it lets us find the step at which two runs
first differ. The scientific goal is weaker and more robust. A conclusion,
such as "the change raises accuracy", should hold across seeds, so we report
the mean and the spread over several runs rather than one run. As with the
number formats of that section, a conclusion that changes with the
seed or with the last bits of the arithmetic needs more evidence.

## Model Inspection with Hooks

A repeatable run tells us that a problem will recur, not where it arises.
To find out, we observe what each layer computes during the run. Editing the
`forward` method of every layer to print or store its output is tedious, and
impossible for a model that came from a library or a checkpoint whose code
we do not want to change. A *hook* is a function that the framework calls at
a fixed point of each layer's computation, from outside the layer's code.

`nnx.capture` wraps a call of a model and records, alongside the output, the
value that every submodule returned, without any change to the model's code.
the figure shows where such an observation point sits in the
computation of each module.

![The `__call__` wrapper as a pipeline: input flows through pre-hooks, then forward, then hooks, to the output, with the two hook stages dashed and orange against forward's solid blue, and a side arrow from the hooks stage to an observer that can capture, check, or modify.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-hooks.svg)

As a test model we use a stack of eight residual blocks of width 64 between
an input layer with 20 inputs and an output layer with 10 outputs. The block
is the residual block of that section, unchanged: it adds
the output of a small network, its *body*, to its input: $\mathbf{h} = \mathbf{x} +
\textrm{body}(\mathbf{x})$, where the body is a linear layer, a ReLU, and a
second linear layer. The weights start from Xavier initialization
(that section) and the biases at zero.

In [5]:
xavier = nnx.initializers.xavier_uniform()  # biases start at zero

class ResidualBlock(nnx.Module):
    def __init__(self, num_hiddens, *, rngs):
        self.body = nnx.Sequential(
            nnx.Linear(num_hiddens, num_hiddens, kernel_init=xavier,
                       rngs=rngs),
            nnx.relu,
            nnx.Linear(num_hiddens, num_hiddens, kernel_init=xavier,
                       rngs=rngs))

    def __call__(self, X):
        return X + self.body(X)

rngs = nnx.Rngs(d2l.get_key())
net = nnx.Sequential(nnx.Linear(20, 64, kernel_init=xavier, rngs=rngs),
                     *[ResidualBlock(64, rngs=rngs) for _ in range(8)],
                     nnx.Linear(64, 10, kernel_init=xavier, rngs=rngs))
X = jax.random.normal(d2l.get_key(), (256, 20))

### Activation Statistics

The initialization experiments of that section measured the
standard deviation of the activations at every depth. With hooks, the same
measurement takes a few lines on an unmodified model. We record the standard
deviation of each layer's output on a minibatch of 256 standard normal
inputs.

In [6]:
_, inter = nnx.capture(
    net, nnx.Intermediate, method_outputs=nnx.Intermediate)(X)
for k, layer in enumerate(net.layers):
    out = inter['layers'][k]['__call__'][0]
    print(f'{type(layer).__name__:15s} std {out.std():.2f}')

Linear          std 0.69
ResidualBlock   std 0.83
ResidualBlock   std 1.03
ResidualBlock   std 1.27
ResidualBlock   std 1.60
ResidualBlock   std 1.98
ResidualBlock   std 2.34
ResidualBlock   std 2.93
ResidualBlock   std 3.70
Linear          std 5.11


The standard deviation grows from block to block, from about 0.7 after the
input layer to roughly 3 to 4 after the eighth block. Each block adds its
body's output to its input, and the two are nearly uncorrelated, so their
variances add. With Xavier initialization the body's output has about half
the variance of its input: the first linear layer keeps the variance, the
ReLU halves the mean square, and the second linear layer keeps it again. The
variance therefore grows by a factor of about 1.5 per block, and the
standard deviation by about $\sqrt{1.5} \approx 1.22$. This growth with
depth is why the residual branches of deep networks are often initialized
with smaller weights (that section).

Nothing needs to be detached or removed: the captured values are ordinary
arrays, the dictionary exists for this one call, and no observer stays
attached to the model. `method_outputs=nnx.Intermediate` records the return
value of every module call. For finer control, a module can call
`self.sow(nnx.Intermediate, 'name', value)` at chosen points in its
computation, and `nnx.capture(model, nnx.Intermediate)` then returns only
those values.

### A NaN Finder

When a loss becomes NaN at step 40,000, the useful question is which layer
produced the first non-finite value. A hook on every layer that checks its
output for infinities and NaNs answers it in one forward pass: the first
hook to fire names the layer, because hooks fire in the order the layers
compute. To test this, we put a NaN into one weight of the fourth layer of
the stack, the third residual block:

In [7]:
saved = net.layers[3].body.layers[0].kernel[0, 0]
net.layers[3].body.layers[0].kernel[0, 0] = float('nan')  # break one layer
_, inter = nnx.capture(
    net, nnx.Intermediate, method_outputs=nnx.Intermediate)(X)

def get_path(tree, path):
    for key in path:
        tree = tree[key]
    return tree

for path, module in nnx.iter_modules(net):
    if isinstance(module, nnx.Linear):
        out = get_path(inter, path)['__call__'][0]
        if not jnp.isfinite(out).all():
            print('first non-finite output in', path)
            break

net.layers[3].body.layers[0].kernel[0, 0] = saved  # restore the weight

first non-finite output in ('layers', 3, 'body', 'layers', 0)


The report names `('layers', 3, 'body', 'layers', 0)`, the layer that holds
the NaN. We checked the captured outputs of the linear layers in the order
of the module tree, which for this sequential network is the order in which
they compute. For a search during training, `jax.config.update('jax_debug_nans',
True)` makes JAX raise an error at the first operation that produces a NaN.

### Gradient Inspection

Gradients can be inspected in the same way, for instance to log the size of
each layer's gradient or to find where the gradients start to explode. As a
loss we take the mean squared output of the stack on a minibatch of 256
standard normal inputs.

JAX needs no backward hooks: `nnx.grad` returns the gradients as a tree of
values with the same structure as the parameters, which we inspect or
transform like any other data. Here are the gradient norms of the first
linear layer of the third residual block:

In [8]:
grads = nnx.grad(lambda model: (model(X) ** 2).mean())(net)
layer = grads['layers'][3]['body']['layers'][0]  # block 3, first layer
print(f"kernel {float(jnp.linalg.norm(layer['kernel'][...])):.3g}, "
      f"bias {float(jnp.linalg.norm(layer['bias'][...])):.3g}")

kernel 26.6, bias 5.3


We can now repeat a run bit for bit on one machine, say which sources of
randomness and which operations stand in the way, and observe any layer of
an unmodified model during a run. What these tools cannot do is make two
different machines agree, or decide on their own whether a difference
between two runs matters; for that we compare results across seeds.

## Summary

Every random draw in a training run comes from a generator fixed by a seed:
initialization, dropout, shuffling, augmentation, and loader workers all
consult one. Seeding each generator the run uses fixes the draws, and private
generator objects or explicit keys keep unrelated draws from shifting them.
Seeding does not fix the arithmetic: floating-point addition depends on the
order of the terms, and some GPU operations add in a varying order, so
bitwise repetition also needs deterministic operations and holds only on one
machine and one software version. It is a tool for debugging; scientific
conclusions should hold across seeds. Hooks, or their equivalents in each
framework, observe the outputs and gradients of every layer of an unmodified
model, which turns questions such as "where did the first NaN appear?" into
one forward pass.

## Exercises

1. [code] **Block ablation.** `sow` records intermediates but cannot change
   them, so an output-replacing hook cannot be written.
    1. Zero the output of one residual block's body by editing
       `ResidualBlock.__call__`, which turns the block into the identity,
       and measure the relative change in `net`'s output on a fixed batch.
       Repeat for every block, and rank the blocks by this change.
    1. Rebuild `net` three more times with new random weights and repeat
       the ranking. State whether the conclusion "block $k$ matters most"
       survives, and what evidence such a claim would need.
2. [code] **Residual variance budget.** Let $\mathbf{x}_k$,
   $\mathbf{b}_k$ and $\mathbf{h}_k$ denote the input, body output and
   output of block $k$, with each variance taken over all entries. Collect
   every layer's output and every body's output from one call of
   `nnx.capture` with `method_outputs=nnx.Intermediate`. The input of
   block $k$ is the output of the layer before it. The standard deviations
   in the table of that section grow because each
   block adds its body's output to the stream. Measure how closely

   $$\textrm{Var}[\mathbf{h}_k] = \textrm{Var}[\mathbf{x}_k] + \textrm{Var}[\mathbf{b}_k]$$

   holds for every block, and name the term this equation omits.
3. [code] **Seeds and data loading.** ● Simulate a four-worker input
   pipeline in plain Python: split a dataset of 64 examples with 20
   features into four shards, one per worker, and let each worker add
   Gaussian noise to its examples with `jax.random.normal`.
    1. Derive the noise keys by splitting one base key into per-worker keys
       with `jax.random.split(key, 4)` and folding the epoch number into
       each with `jax.random.fold_in`. Verify that two runs from the same
       base key produce identical noise. Then reduce the number of workers
       to two and check whether each example still receives the same noise.
    1. Derive each example's key instead by folding the epoch number into
       the base key and then folding in the example's index. Repeat both
       checks, and explain which derivation makes an example's noise
       independent of how examples are assigned to workers.
4. [code] **Divergence monitor.** ● Turn the finiteness check of
   that section into a per-step training monitor. Rebuild
   `net` and train it with SGD at learning rate 1 on the mean squared
   error between its output and standard normal targets, using one fixed
   batch of 256 standard normal inputs. Report the first step at which a
   layer's output is non-finite, and the first such layer by depth. Check
   the loss at every step as well, and report whether it becomes
   non-finite before any layer output does.
5. [code] **Gradient norms.** ● Rebuild `net`, multiply the kernel of
   every linear layer inside its residual blocks by 3, and use the mean
   squared output on a fixed batch as the loss.
    1. Compute per-block parameter-gradient norms from the tree that
       `nnx.grad` returns. To obtain each block's *output* gradient, split
       the forward computation at that block and differentiate the tail
       with respect to the block's output. Plot both families of norms
       against depth.
    1. Express the gradient of a linear layer's kernel through the layer's
       input and its output gradient. Use this expression to explain why
       the two depth profiles differ, and state which family to monitor for
       an exploding backward signal.
6. [code] **Capture coverage.** Find the value computed inside each
   block's `body` that a call of `nnx.capture` with
   `method_outputs=nnx.Intermediate` does not record, and explain why.
   Edit `ResidualBlock` so that it records the value with `self.sow`, and
   collect it with `nnx.capture(net, nnx.Intermediate)`.

[Discussions](https://d2l.discourse.group/t/36759)